# Notebook 07 — Genetic Drift, Mutation, and Migration

**Module:** 05 — Biology Fundamentals  
**Notebook:** 07 of 18  
**Prerequisites:** NB06  
**Time estimate:** 90 minutes

> The Wright-Fisher model implemented here is the foundation of **NB15 (Mini-Project)**.
> Understand the code in this notebook before attempting the mini-project.

---
## Step 1 — Motivation

Not all allele frequency changes are driven by natural selection. In small populations,
random sampling alone causes allele frequencies to wander — a phenomenon called **genetic
drift**. Understanding drift is essential for interpreting population genomics: without it,
you might mistake a drifted allele for a selected one. The neutral theory of evolution
(Kimura, 1968) holds that most genomic variation is driven by drift, not selection.

---
## Step 2 — Intuition

Imagine a population of just 10 individuals. Each generation, you randomly choose
20 alleles (2 per individual) from the previous generation to form the new one.
Even if A and a start at 50/50, pure luck can make A reach 100% (fixation) or 0%
(loss) within a few dozen generations. This is drift — randomness due to finite
population size. In a population of 1,000,000, the same randomness averages out
and allele frequencies move much more slowly.

---
## Step 3 — Biological Background

**Three forces (besides selection) that change allele frequencies:**

1. **Genetic drift:** random sampling in finite populations. Effect: alleles are
   randomly lost or fixed. Stronger in small populations. The founder effect and
   population bottleneck are special cases.

2. **Mutation:** creates new alleles at rate μ per base per generation.
   In humans, μ ≈ 1.2 × 10⁻⁸ per base per generation. Mutation alone changes
   allele frequency very slowly (one new mutation in a population of N changes
   the frequency by 1/(2N)).

3. **Migration (gene flow):** movement of individuals (and their alleles) between
   populations. Tends to homogenise allele frequencies between populations.
   Population genetics uses Fst (fixation index) to measure how differentiated
   two populations are — Fst = 0 means identical allele frequencies; Fst = 1 means
   completely different.

**The Wright-Fisher model:**
- Population size N is constant each generation
- In generation t, allele frequency is p_t
- In generation t+1, the number of A alleles is drawn from Binomial(2N, p_t)
- p_{t+1} = count / (2N)
- This is the simplest model capturing drift; selection can be added by modifying p
  before sampling

**Expected time to fixation:**
For a neutral allele starting at frequency p₀ in a population of size N:
- Expected time to fixation | fixed ≈ −4N·ln(p₀) / (1 − p₀) generations
- Probability of eventual fixation from frequency p₀ = p₀
- Average time to fixation for a new neutral mutation (p₀ = 1/2N) ≈ 4N generations

---
## Step 4 — Mathematical Explanation

**Wright-Fisher sampling:**
$$k \sim \text{Binomial}(2N,\, p_t)$$
$$p_{t+1} = \frac{k}{2N}$$

**Variance of allele frequency change per generation:**
$$\text{Var}(\Delta p) = \frac{p(1-p)}{2N}$$

This shows that drift is stronger (higher variance) when:
- N is small, or
- p is near 0.5 (maximum heterozygosity)

**Effective population size (N_e):**
Real populations don't breed randomly or maintain constant size. N_e is the size of
an ideal Wright-Fisher population that would exhibit the same drift rate. For humans,
N_e ≈ 10,000 — much smaller than the current 8 billion.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Wright-Fisher simulation (drift only)
def wright_fisher_simulation(
    N: int,
    p0: float,
    n_gen: int,
    n_trials: int,
    s: float = 0.0,
    mu: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """
    Simulate Wright-Fisher genetic drift with optional selection and mutation.

    Parameters
    ----------
    N        : diploid population size
    p0       : initial frequency of allele A
    n_gen    : number of generations to simulate
    n_trials : number of independent populations
    s        : selection coefficient (0 = neutral; positive favours A)
    mu       : mutation rate (A → a per generation, symmetric)

    Returns
    -------
    trajectories : np.ndarray of shape (n_trials, n_gen+1)
                   Each row is one population's allele-frequency trajectory.
    """
    rng = np.random.default_rng(seed)
    trajectories = np.zeros((n_trials, n_gen + 1))
    trajectories[:, 0] = p0

    for t in range(n_gen):
        p = trajectories[:, t]

        # Selection: increase frequency of A relative to a
        # Fitness: w_AA = (1+s)², w_Aa = (1+s), w_aa = 1 (multiplicative)
        # Mean fitness: w_bar = p²(1+s)² + 2pq(1+s) + q²
        # After selection: p' = p(1+s) / w_bar  [additive approximation]
        if s != 0.0:
            q = 1.0 - p
            p_sel = p * (1 + s) / (p * (1 + s) + q)
        else:
            p_sel = p

        # Mutation (symmetric: A→a and a→A at rate mu)
        if mu != 0.0:
            p_sel = p_sel * (1 - mu) + (1 - p_sel) * mu

        # Genetic drift: binomial sampling
        counts = rng.binomial(2 * N, p_sel)
        trajectories[:, t + 1] = counts / (2 * N)

    return trajectories

# Basic sanity check
traj = wright_fisher_simulation(N=100, p0=0.5, n_gen=200, n_trials=50)
fixed = (traj[:, -1] >= 1.0).sum()
lost  = (traj[:, -1] <= 0.0).sum()
print(f"N=100, p0=0.5, 200 generations, 50 trials")
print(f"  Fixed (p=1): {fixed} trials  ({fixed/50:.0%})")
print(f"  Lost  (p=0): {lost} trials  ({lost/50:.0%})")
print(f"  Still segregating: {50 - fixed - lost} trials")

In [ ]:
# Cell 6.2 — Effect of population size on drift speed
pop_sizes = [10, 50, 200, 1000]
results = {}
for N in pop_sizes:
    t = wright_fisher_simulation(N=N, p0=0.5, n_gen=500, n_trials=100)
    results[N] = t

print("Population size effect (p0=0.5, 500 generations, 100 trials):")
print(f"  {'N':<6} {'Fixed':>8} {'Lost':>8} {'Mean final p':>14}")
for N, t in results.items():
    fixed = (t[:, -1] >= 1.0).sum()
    lost  = (t[:, -1] <= 0.0).sum()
    print(f"  {N:<6} {fixed:>8} {lost:>8} {t[:,-1].mean():>14.3f}")

In [ ]:
# Cell 6.3 — Selection vs. drift comparison
# Neutral drift
neutral = wright_fisher_simulation(N=200, p0=0.1, n_gen=300, n_trials=50, s=0.0)
# Positive selection (s=0.05: 5% fitness advantage)
selected = wright_fisher_simulation(N=200, p0=0.1, n_gen=300, n_trials=50, s=0.05)

print("Neutral drift vs. positive selection (N=200, p0=0.1, 300 generations):")
print(f"  Neutral:   final mean p = {neutral[:,-1].mean():.3f}, fixed = {(neutral[:,-1]>=1).sum()}")
print(f"  Selection: final mean p = {selected[:,-1].mean():.3f}, fixed = {(selected[:,-1]>=1).sum()}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Drift trajectories: small vs. large population + selection effect
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Drift trajectories for different population sizes
ax = axes[0]
colors_pop = {10: 'tomato', 50: 'orange', 200: 'steelblue', 1000: 'seagreen'}
gen_axis = np.arange(501)
for N, color in colors_pop.items():
    t = results[N]
    # Plot first 10 trajectories per N
    for trial in t[:10]:
        ax.plot(gen_axis, trial, color=color, alpha=0.3, lw=0.7)
    ax.plot([], [], color=color, label=f'N={N}', lw=2)  # legend entry

ax.set_xlabel('Generation'); ax.set_ylabel('Allele frequency p')
ax.set_title('Genetic drift: allele frequency trajectories\n(10 trials per population size)')
ax.legend(title='Pop. size', fontsize=9)
ax.axhline(0.5, color='gray', ls='--', lw=0.8)

# Panel 2: Selection rescues allele from drift
ax = axes[1]
gen_axis_300 = np.arange(301)
for trial in neutral[:15]:
    ax.plot(gen_axis_300, trial, color='steelblue', alpha=0.3, lw=0.8)
for trial in selected[:15]:
    ax.plot(gen_axis_300, trial, color='tomato', alpha=0.3, lw=0.8)
ax.plot([], [], color='steelblue', label='Neutral (s=0)', lw=2)
ax.plot([], [], color='tomato', label='Selection (s=0.05)', lw=2)
ax.set_xlabel('Generation'); ax.set_ylabel('Allele frequency p')
ax.set_title('Selection vs. drift (N=200, p0=0.1)\nSelection drives allele toward fixation')
ax.legend()
ax.axhline(0.1, color='gray', ls='--', lw=0.8, label='p0')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `wright_fisher_simulation(N, p0, n_gen, n_trials)` from scratch
   (drift only, no selection, no mutation). Verify that with N=50, p0=0.5,
   and 1000 generations, roughly 50% of trials end in fixation and 50% in loss.
2. A population of N=20 undergoes a bottleneck. Starting at p0=0.5, simulate
   50 trials for 500 generations. How many alleles are lost or fixed?
   Compare to N=1000 with the same p0.
3. What is Kimura's neutral theory? Why would the observation that most DNA sequence
   variants have very small effects support it?
4. Fst (fixation index) measures population differentiation. If two populations
   have allele frequencies p₁=0.8 and p₂=0.2 at a locus, and the average frequency
   is p̄=0.5, Fst = Var(p) / (p̄(1−p̄)). Calculate Fst for this example.

---
## Quiz — Active Recall

1. What is genetic drift? Why is it stronger in small populations?
2. State the Wright-Fisher model in one sentence.
3. What is the probability that a new neutral mutation eventually reaches fixation?
4. What is the founder effect? Give a real-world example.
5. How does migration (gene flow) affect allele frequencies between two populations?

---
## Papers Referenced

- Kimura (1968). DOI: 10.1038/217624a0 (Evolutionary rate at the molecular level)

---
## Reflection

**Date completed:** ____________________

> *[Can you explain why the effective population size of humans is ~10,000 even
> though there are 8 billion people alive today? What does this number actually measure?]*

---
*Next: `08_natural_selection_fitness_models.ipynb`*